## 1. 模型调用方式

### 1.1 消息类型：传什么

#### 1.1.1 基础示例

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage,ToolMessage
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(model="deepseek-v4-flash",
                api_key=os.getenv("DEEPSEEK_API_KEY"),
                base_url=os.getenv("DEEPSEEK_BASE_URL")
                )
response = llm.invoke([HumanMessage(content="你好，你能帮我做什么事情?")])
print(response.content)

#### 1.1.2 构建对话历史

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage,ToolMessage
# 对话历史
conversation = [
    SystemMessage(content="你是一个有帮助的AI助手"),
    HumanMessage(content="你好，我叫hzk"),
    AIMessage(content="你好！hzk,有什么我可以帮助你的吗？"),
    HumanMessage(content="我叫什么名字？"),
]

response = llm.invoke(conversation)
print(response.content)

###  1.2 传入方式：怎么传

#### 1.2.1 直接传入字符串（最简单）

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="deepseek-v4-flash",
                api_key=os.getenv("DEEPSEEK_API_KEY"),
                base_url=os.getenv("DEEPSEEK_BASE_URL")
                )
# 直接传入字符串
response = llm.invoke("你好，介绍一下LangChain")
print(response.content)

#### 1.2.2 传入消息列表（最常用）

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage,ToolMessage
llm = ChatOpenAI(model="deepseek-v4-flash",
                api_key=os.getenv("DEEPSEEK_API_KEY"),
                base_url=os.getenv("DEEPSEEK_BASE_URL")
                )
# 传入消息列表
# invoke能传入的参数类型：
# 1. PromptValue | str | Sequence[MessageLikeRepresentation]
# 2. BaseMessage | list[str] | tuple[str, str] | str | dict[str, Any]
response = llm.invoke(
    [
        SystemMessage(content="你是一个专业的AI工程师"),
        HumanMessage(content="你好，介绍一下LangChain")
    ]
)
print(response.content)

In [5]:
from langchain_core.messages import HumanMessage, AIMessage

# 对话历史
conversation = [
    HumanMessage(content="什么是LangChain？"),
    AIMessage(content="LangChain是一个用于开发大模型应用的框架。"),
    HumanMessage(content="它有哪些核心组件？")  # 这依赖于上一轮的上下文
]

response = llm.invoke(conversation)
print(response.content)

LangChain的核心组件可以理解为搭建大模型应用的“乐高积木”。主要可以分为以下几块：

1.  **模型 I/O （Model I/O）**：与大模型交互的基础模块。
    -   **Prompts（提示）**：管理、模板化和动态选择提示词（Prompt），让你的提问更规范、更有效。
    -   **LLMs / Chat Models（大语言模型/对话模型）**：统一接口，方便你切换不同的底层模型（如OpenAI、Claude、本地开源模型）。
    -   **Output Parsers（输出解析器）**：将模型返回的文本结构化，例如提取JSON、列表或自定义格式的数据。

2.  **检索增强生成 （RAG）**：让模型能访问你私有的或实时的数据。
    -   **Document Loaders（文档加载器）**：从PDF、网页、数据库等100多种数据源加载文档。
    -   **Text Splitters（文本分割器）**：将长文档切分成适合模型处理的块（Chunks）。
    -   **Vector Stores（向量存储）**：将文档转为向量并存储，以便快速搜索相似内容（如Chroma、Pinecone）。
    -   **Embedding Models（嵌入模型）**：将文本转换为向量表示的模型。

3.  **Chains（链）**：核心编排模块，将多个步骤组合成一个流水线。
    -   例如：`链 = 提示词模板 + 模型 + 输出解析器`。
    -   你可以创建简单的顺序链（LLMChain），也可以构建复杂的分支或并行链（如`Router Chain`、`Parallel Chain`）。

4.  **Agents（代理）**：让模型能自主决策并调用工具。
    -   **Tool（工具）**：允许模型调用的外部功能，如搜索、计算器、API请求、数据库查询。
    -   **Agent Executor**：核心循环——模型思考“该用什么工具”，执行工具获取结果，再基于结果决定下一步，直到完成任务。

5.  **Memory（记忆）**：让对话或任务具有上下文。
    -   管理聊天历史、状态或实体信息。例如`ConversationBufferMemory`（完整记录）、`Co

#### 1.2.3 使用元组或字典（最灵活）

In [7]:
# # 元组方式：(角色, 内容)
tuple_message = [("system", "你是一个专业的AI工程师"), ("user", "什么是LangGraph?")]
# # 字典方式：{"role": 角色, "content": 内容}
dict_message = [{"role": "system", "content": "你是一个专业的AI工程师"},
                {"role":"user", "content": "什么是LangGraph?"}]

print(llm.invoke(tuple_message).content)
print(llm.invoke(dict_message).content)

LangGraph 是 LangChain 生态中的一个核心库（用 Python 编写），专门用于构建**有状态、多参与者**的 AI 应用程序，特别是需要**复杂控制流**的语言模型代理（Agent）工作流。

简单来说，它用**有向图**（Directed Graph）来编排大语言模型（LLM）、工具函数、人类反馈等组件之间的交互。

### 核心概念
- **State Graph（状态图）**：整个图维护一个**共享状态**（State），所有节点都可以读取和写入该状态。
- **Node（节点）**：每个节点代表一个处理单元，比如调用 LLM、执行工具、处理用户输入等。节点输入输出都是状态字典。
- **Edge（边）**：连接节点，决定数据流动方向。
- **Conditional Edge（条件边）**：根据当前状态动态选择下一个节点（例如，LLM 判断是否需要继续调用工具，或者结束循环）。
- **循环（Cycles）**：LangGraph 原生支持有向环，这是构建代理（如 ReAct、Plan-and-Execute）的关键——让 LLM 可以反复思考、调用工具、再思考，直到得到最终答案。

### 为什么需要 LangGraph？
传统的 LLM 链（Chain）是线性或简单分叉的，无法处理：
- **循环/迭代**（如多次工具调用后修正答案）
- **分支与合并**（并行执行多个子任务）
- **持久化状态**（在多次交互中保持记忆）
- **人类介入**（在 Agent 流程中间暂停，等待人工审核或输入）

LangGraph 通过图中的**节点暂停（Interrupt）**、**状态持久化**和**条件循环**解决了这些问题，特别适合：
- 复杂的 **Agent 循环**（ReAct、OpenAI Functions Agent）
- **多步推理**（如自行分解问题、搜索、验证）
- **人机协作**工作流（Agent 做大部分工作，关键决策点请求人类确认）

### 一个简单示例（伪代码）
```python
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class AgentState(TypedDict):
 

In [8]:
prompt_template = [{"role": "system", "content": "你是一个{role}"}, 
                    {"role": "user", "content": "请帮我解释{topic}"}]
# 动态填充
message = [{"role": t["role"], 
            "content": t["content"].format(role = "机器学习", topic = "过拟合")} 
            for t in prompt_template]
            
print(llm.invoke(message).content)

当然。过拟合（Overfitting）是机器学习中一个非常常见的问题，简单来说就是**模型在训练数据上表现很好，但在新的、未见过的数据上表现很差**。就像学生只会做做过的原题，稍微变个题型就懵了。

### 🔍 直观理解
- **训练集**：你给模型看的“课本和作业”。  
- **测试集/新数据**：真正的“考试题目”。  

过拟合的模型把训练数据里的噪声、偶然模式甚至错误都记住了，而不是学习真正的规律。结果就是：训练误差非常低（甚至为零），但测试误差很高。

### 📌 常见原因
1. **模型太复杂**：比如决策树深度过大、神经网络层数过多、多项式回归次数过高，导致模型有太多参数可以“死记硬背”数据。  
2. **训练数据太少**：数据量不足以代表真实分布，模型容易把局部特征当成全局规律。  
3. **噪声过多**：数据中有明显的错误或异常值，模型强行去拟合它们。  
4. **训练时间过长**：比如神经网络迭代太多轮次，模型反复学习噪声。

### ⚠️ 典型表现
- 训练集准确率接近100%，但验证集/测试集准确率很低。  
- 学习曲线中，训练损失持续下降，验证损失先降后升（出现拐点）。  
- 模型对输入数据的微小变化非常敏感（比如图片加一个像素就预测错误）。

### 🛠 如何检测？
最常用方法：**交叉验证**。把数据分成训练集和验证集，观察两者性能差距。如果训练误差远小于验证误差，基本就是过拟合了。

### ✅ 如何避免？
| 方法 | 说明 | 例子 |
|------|------|------|
| **增加数据量** | 更多样本让模型学到通用模式 | 数据增强（图像翻转、裁剪） |
| **降低模型复杂度** | 减少参数或使用更简单的模型 | 减少决策树深度，降低多项式次数 |
| **正则化** | 对过大权重施加惩罚 | L1/L2正则化、Dropout（神经网络） |
| **早停** | 验证误差不再下降时停止训练 | 监控验证损失，在拐点处停止 |
| **集成学习** | 用多个模型投票，减少单模型过拟合风险 | 随机森林、Bagging |
| **数据清洗** | 去除异常值和噪声 | 检查数据分布，剔除明显错误 |

### 🔁 举一个简单例子
你想用多项式拟合一些点：  
- 真实规律是二次曲线（抛物线）。

### 1.3. 调用方式：怎么调

#### 1.3.1 同步调用 - invoke()（最常用）

In [ ]:
response = llm.invoke("什么是LangChain？")
print(response.content)

#### 1.3.2  异步调用 - ainvoke()（高并发）

In [ ]:
import asyncio
from langchain_openai import ChatOpenAI

async def call_llm_async():
    response = await llm.ainvoke("什么是langchain?")
    print(response.content)

# Jupyter Notebook 中直接 await
await call_llm_async() # 方式一
# asyncio.run(call_llm_async()) # 方式二

LangChain 是一个用于构建基于大语言模型（LLM）的应用程序的**开源开发框架**。你可以把它理解为一套“乐高积木”和“工具包”，它解决了直接调用大模型 API 时经常会遇到的一些复杂问题，比如：

-   **连接外部数据**：让模型能读取你的 PDF、数据库或网页内容（比如做知识库问答）。
-   **执行多步任务**：让模型不仅能聊天，还能调用搜索引擎、执行代码、发邮件等（比如一个能自动查天气、订酒店的智能助手）。
-   **管理对话记忆**：处理长对话时，只记住关键信息，避免超出模型的上下文窗口限制。
-   **搭建复杂流程**：将多个步骤串联起来，形成“链”（Chain）或“智能体”（Agent）。

**简单来说：**

如果大语言模型（如 GPT-4）是一台**强大的引擎**，那么 LangChain 就是**为这台引擎设计的底盘、方向盘和导航系统**，让你能把它组装成一辆能跑会动的汽车（应用程序），而不是光有引擎放在那里。

**它包含几个核心模块：**

1.  **模型 I/O**：统一接口调用不同厂商的模型（OpenAI、Claude、本地模型等），并方便地管理提示词（Prompt）。
2.  **检索增强生成（RAG）**：最核心的用途之一。把你的文档切碎、向量化存起来，当用户提问时，先搜索相关片段，再把这些片段和问题一起发给模型，让模型“看完资料再回答”，极大减少幻觉。
3.  **链（Chains）**：把多个操作（如“调用模型 → 格式化输出 → 再调用另一个模型”）组合成一个可重复使用的流水线。
4.  **智能体（Agents）**：让模型自己决定下一步做什么（例如：用户问“明天天气如何”，模型自动调用天气 API 获取数据，再生成回答）。
5.  **内存（Memory）**：维护对话历史，支持多种记忆策略。

**一个典型的场景：**

> **使用 LangChain 构建一个“公司内部知识库问答机器人”**
>
> 1.  把公司所有的 Word/PDF 文档导入。
> 2.  用 LangChain 的文档加载器读入，文本分割器切块。
> 3.  将切好的文本块向量化存入向量数据库（如 Chroma）。
> 4.  用户提问：“去年的年度总结里提到的新市场战略是什么？”
> 5.  LangChain 在数据库

In [12]:
import time
import asyncio
from langchain_openai import ChatOpenAI


# 准备 5 个测试问题
prompts = [
    "用一句话介绍一下北京",
    "用一句话介绍一下上海",
    "用一句话介绍一下广州",
    "用一句话介绍一下深圳",
    "用一句话介绍一下杭州"
]


# ========== 测试一：同步 invoke（串行） ==========
def test_sync_invoke():
    print("=== 同步 invoke ===")
    start_time = time.time()

    for i, prompt in enumerate(prompts):
        print(f"  [同步] 正在发送第 {i + 1} 个请求...")
        llm.invoke(prompt)  # 死等，拿到结果才进入下一次循环

    print(f"总耗时: {time.time() - start_time:.2f} 秒\n")


# ========== 测试二：异步 ainvoke（并行） ==========
async def test_async_ainvoke():
    print("=== 异步 ainvoke ===")
    start_time = time.time()

    # 关键：用 asyncio.gather 同时派发所有请求
    print("  [异步] 瞬间派发 5 个请求...")
    tasks = [llm.ainvoke(prompt) for prompt in prompts]
    results = await asyncio.gather(*tasks)

    for r in results:
        print(f"  回答: {r.content[:20]}...")

    print(f"总耗时: {time.time() - start_time:.2f} 秒\n")


# ========== 运行对比 ==========
async def main():
    test_sync_invoke()         # 先跑同步
    await test_async_ainvoke() # 再跑异步

await main()

=== 同步 invoke ===
  [同步] 正在发送第 1 个请求...
  [同步] 正在发送第 2 个请求...
  [同步] 正在发送第 3 个请求...
  [同步] 正在发送第 4 个请求...
  [同步] 正在发送第 5 个请求...
总耗时: 12.39 秒

=== 异步 ainvoke ===
  [异步] 瞬间派发 5 个请求...
  回答: 北京是中华人民共和国的首都，是全国政治中...
  回答: 上海是中国最大的经济中心和国际化大都市，...
  回答: 广州是中国南方的国家中心城市和综合性门户...
  回答: 深圳是中国首个经济特区，以高速发展的科技...
  回答: 杭州是一座融合了千年历史底蕴与数字经济活...
总耗时: 3.05 秒



#### 1.3.3 流式调用 - stream()（打字机效果）

#### 1.3.4 批次调用 - batch()（并行处理）

In [ ]:
def batch_example():
    from langchain_openai import ChatOpenAI
    
    questions = [
        "什么是Python？",
        "什么是JavaScript？",
        "什么是Go语言？"
    ]

    responses = llm.batch(questions)
    for q, r in zip(questions, responses):
        print(f"Q: {q}")
        print(f"A: {r.content}\n")

batch_example()

In [ ]:
async def batch_async():
    questions = [
        "什么是LangChain？",
        "LangChain的核心组件有哪些？",
        "如何使用LangChain构建Agent？"
    ]
    responses = await llm.abatch(questions)
    for q, r in zip(questions, responses):
        print(f"Q: {q}\nA: {r.content}\n")

await batch_async()

###  1.4 调用配置与高级特性

#### 1.4.1 运行时配置

#### 1.4.2 回调处理器

#### 1.4.3 运行时动态切换模型